# LeniencyBench — Training Notebook

End-to-end SFT pipeline against the **LeniencyBench** OpenEnv environment ([live env](https://huggingface.co/spaces/shreyas-garg/drift-env)).

---

## ⚠️ Read this first — what this notebook produces

This notebook orchestrates `train.py` (the actual training script). What it produces depends on the `QUICK_MODE` flag set in cell 3:

| Mode | Model | Compute | Time | What you get |
|---|---|---|---|---|
| `QUICK_MODE='true'` (default) | Qwen 2.5 **0.5B** | Colab T4 (free) | ~10 min | Pipeline correctness check; small-scale validation. Drift-sens acc moves 0% → 50%. |
| `QUICK_MODE='false'` | Qwen 2.5 **3B** | A100/A10G | ~100 min | **The headline 91.3% tightening result.** Requires real GPU compute (HF Jobs, Colab Pro, etc). |

## 🎯 The headline 91.3% tightening result was produced by this exact same script with `QUICK_MODE='false'` on Qwen 2.5 3B, executed via HF Jobs on an A100-SXM4-80GB.

**Verifiable artifacts from that 3B run** (publicly accessible):

- 📊 **Trained LoRA adapter + training logs:** [huggingface.co/shreyas-garg/leniencybench-qwen3b-outputs](https://huggingface.co/shreyas-garg/leniencybench-qwen3b-outputs)
- 📜 **Raw stdout from the 3B run** (incl. SFT loss per step, pre/post-SFT eval blocks): [`outputs/v7_full_logs.txt`](https://github.com/shreyas-garg/OpenEnv/blob/main/outputs/v7_full_logs.txt)
- 📈 **SFT loss curve plot:** [`outputs/sft_loss.png`](https://github.com/shreyas-garg/OpenEnv/blob/main/outputs/sft_loss.png)
- 📊 **Direction-split bar chart:** [`outputs/direction_split.png`](https://github.com/shreyas-garg/OpenEnv/blob/main/outputs/direction_split.png)
- 🗂️ **JSON eval snapshots:** [`outputs/evals_v7.json`](https://github.com/shreyas-garg/OpenEnv/blob/main/outputs/evals_v7.json)
- 🗂️ **JSON SFT log (per-step loss):** [`outputs/sft_log_v7.json`](https://github.com/shreyas-garg/OpenEnv/blob/main/outputs/sft_log_v7.json)

## Pipeline at a glance
1. Install deps (Unsloth + TRL 0.24)
2. Clone the repo
3. Pick run mode (QUICK_MODE flag)
4. Sanity-check the env + dataset
5. Run `train.py` end-to-end (SFT + GRPO try/except)
6. Inspect saved LoRA adapters

## 1. Install dependencies

Unsloth needs `trl>=0.18.2`. Do NOT pin older TRL or the install breaks.

In [ ]:
!pip install --upgrade -q pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U "trl==0.24.0" "datasets<4.4.0" accelerate peft bitsandbytes matplotlib
# IMPORTANT: restart runtime AFTER this cell finishes, BEFORE running anything below.
# (Runtime -> Restart runtime)

## 2. Clone the repo (idempotent)

Run this AFTER restarting the runtime. Safe to re-run — won't re-clone.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/shreyas-garg/OpenEnv.git'
REPO_DIR = '/content/OpenEnv'

if not os.path.isdir(os.path.join(REPO_DIR, 'drift_env')):
    if os.path.isdir(REPO_DIR):
        !rm -rf {REPO_DIR}
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())
print('drift_env present:', os.path.isdir('drift_env'))
print('train.py present:', os.path.isfile('train.py'))

## 3. Pick run mode

Default below is **QUICK_MODE='true'** — so a judge can hit "Run all" on a free Colab T4 and see the pipeline execute end-to-end in ~10 minutes on Qwen 2.5 0.5B.

**To reproduce the headline 91.3% tightening number:** flip the line below to `'false'` and run on an A100/A10G (HF Jobs, Colab Pro, RunPod, etc). This will take ~100 minutes on A100. The exact `train.py` script + config used for the headline result is on `main` branch — no other knobs to set.

In [ ]:
os.environ['QUICK_MODE'] = 'true'   # set to 'false' on real GPU compute to reproduce the 3B headline result
os.environ['USE_WANDB'] = 'false'   # set to 'true' + set WANDB_API_KEY for live logging

# To push training outputs to a HF Hub model repo, set these (else outputs stay local):
# os.environ['HF_TOKEN'] = '<your_hf_token_with_write>'
# os.environ['HUB_REPO_ID'] = '<your_username>/leniencybench-outputs'

print(f"QUICK_MODE={os.environ['QUICK_MODE']}")

## 4. Sanity check — env + dataset work

In [ ]:
from drift_env.dataset import build_dataset, dataset_stats
rows = build_dataset(n_episodes=10, start_seed=0)
print(dataset_stats(rows))
print('\nSample prompt (first 400 chars):\n' + rows[5]['prompt'][:400])

## 5. Run the full pipeline

Output will show: pre-training eval → SFT → post-SFT eval → GRPO (try/except) → post-GRPO eval.

**Key number to watch: `tightening` accuracy at each eval stage.**

- In `QUICK_MODE='true'` (0.5B): expect 0% → ~30–50%
- In `QUICK_MODE='false'` (3B): expect **0% → 91.3%** (matches the headline result documented in [outputs/v7_full_logs.txt](https://github.com/shreyas-garg/OpenEnv/blob/main/outputs/v7_full_logs.txt))

In [ ]:
!python train.py

## 6. Inspect saved LoRA adapters

After the run, the adapter lives in `./outputs/lora_adapters/`. Do NOT naively merge 4-bit base → 16-bit + adapter (Unsloth warning); use the dedicated merge path if you need a single checkpoint.

In [ ]:
!ls -la outputs/lora_adapters/ 2>/dev/null || echo 'train.py did not finish — check the run output above'

## 7. Verify the 3B headline result without re-running training

If you don't want to spend ~100 min reproducing the 3B run, you can **load the artifacts from our published HF model repo** and verify the eval numbers directly. This is the fastest way for a reviewer to confirm the headline 91.3% number is real.

Run the cell below — it downloads the published `evals.json` (the eval snapshot from the actual 3B training run) and prints the pre / post-SFT direction-split table.

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
import json

evals_path = hf_hub_download(
    repo_id='shreyas-garg/leniencybench-qwen3b-outputs',
    filename='evals.json',
    repo_type='model',
)
data = json.load(open(evals_path))

print('=== LeniencyBench: Qwen 2.5 3B onsite SFT result ===\n')
print(f"{'Stage':<14} {'tightening':>15} {'loosening':>15} {'overall':>15}")
print('-' * 60)
for stage, name in [('pre', 'pre-training'), ('post_sft', 'post-SFT')]:
    e = data[stage]
    by_dir = e['drift_acc_by_direction']
    overall = e['drift_acc']
    print(f"{name:<14} {by_dir.get('tightening', 0)*100:>14.1f}% {by_dir.get('loosening', 0)*100:>14.1f}% {overall*100:>14.1f}%")

print('\nFull eval JSON saved to:', evals_path)

## Reference: scaling beyond Qwen 2.5 3B

The same `train.py` works for any Unsloth-supported model. Override `MODEL_NAME` to bump up:
- `unsloth/Qwen2.5-7B-Instruct` (needs A100 80GB+)
- `unsloth/Llama-3.2-3B-Instruct` (similar to Qwen 3B)

Set via env var before running cell 5:
```python
os.environ['MODEL_NAME'] = 'unsloth/Qwen2.5-7B-Instruct'
```

Precision auto-detects: bf16 on Ampere+ (A100/H100/A10G), fp16 on Turing (T4).